In [2]:
from basic_transforms import transform_basic
from merchants_risk_encoder import MerchantRiskEncoder
from woe_iv import WOEProcessor
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv('data.csv')

In [3]:
df = pd.read_csv('data.csv')

df = transform_basic(df)

df["target"] = (df["amount_outstanding_21d"] > 0).astype(int)
df = df.drop(columns=["amount_outstanding_21d", "amount_outstanding_14d"])

In [4]:
# time train test split

df = df.sort_values("loan_issue_date")
split_date = df["loan_issue_date"].quantile(0.8)

train = df[df["loan_issue_date"] <= split_date].copy()
test = df[df["loan_issue_date"] > split_date].copy()

target = "target"

X_train = train.drop(columns=[target, "loan_issue_date"])
y_train = train[target]

X_test = test.drop(columns=[target, "loan_issue_date"])
y_test = test[target]

In [5]:
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

target
0    0.943054
1    0.056946
Name: proportion, dtype: float64
target
0    0.955453
1    0.044547
Name: proportion, dtype: float64


In [6]:
merchant_encoder = MerchantRiskEncoder(alpha_group=50)
merchant_encoder.fit(X_train, y_train)

X_train = merchant_encoder.transform(X_train)
X_test = merchant_encoder.transform(X_test)

In [7]:
# woe transformation

num_cols = [
"loan_amount",
"existing_klarna_debt",
"days_since_first_loan",
"num_active_loans",
"new_exposure_7d",
"new_exposure_14d",
"num_confirmed_payments_3m",
"num_confirmed_payments_6m",
"num_failed_payments_3m",
"num_failed_payments_6m",
"num_failed_payments_1y",
"amount_repaid_14d",
"amount_repaid_1m",
"amount_repaid_3m",
"amount_repaid_6m",
"amount_repaid_1y",
"debt_to_loan_ratio",
"exposure_growth",
"failed_payment_ratio_3m",
"failed_payment_ratio_6m",
"repayment_speed",
"repayment_ratio_6m",
"loan_intensity",
"merchant_default_rate",
"merchant_group_default_rate"
]

woe_processor = WOEProcessor(num_cols=num_cols,max_bins=10)

X_train = woe_processor.fit_transform(X_train, y_train)

X_test = woe_processor.transform(X_test)

In [8]:
X_train = X_train.drop(columns=num_cols)
X_test = X_test.drop(columns=num_cols)

bin_cols = [f"{c}_bin" for c in num_cols]

X_train = X_train.drop(columns=bin_cols)
X_test = X_test.drop(columns=bin_cols)

In [9]:
iv_df = pd.DataFrame({
    "feature": woe_processor.iv_dict.keys(),
    "IV": woe_processor.iv_dict.values()
}).sort_values("IV", ascending=False)

print(iv_df)

                        feature        IV
21           repayment_ratio_6m  0.126707
15             amount_repaid_1y  0.110121
23        merchant_default_rate  0.106265
7     num_confirmed_payments_6m  0.103016
14             amount_repaid_6m  0.096216
6     num_confirmed_payments_3m  0.084908
2         days_since_first_loan  0.082153
13             amount_repaid_3m  0.070862
20              repayment_speed  0.065131
12             amount_repaid_1m  0.039630
24  merchant_group_default_rate  0.039421
11            amount_repaid_14d  0.034431
22               loan_intensity  0.029073
16           debt_to_loan_ratio  0.024595
3              num_active_loans  0.015443
0                   loan_amount  0.015305
1          existing_klarna_debt  0.012685
5              new_exposure_14d  0.001689
4               new_exposure_7d  0.000034
10       num_failed_payments_1y  0.000000
9        num_failed_payments_6m  0.000000
8        num_failed_payments_3m  0.000000
17              exposure_growth  0

In [14]:
weak_woe_features = [f"{col}_woe" for col in iv_df.loc[iv_df["IV"] < 0.01, "feature"]]

X_train.drop(columns=weak_woe_features, inplace=True, errors="ignore")
X_test.drop(columns=weak_woe_features, inplace=True, errors="ignore")

print("Dropped weak IV features:", weak_woe_features)

Dropped weak IV features: ['new_exposure_14d_woe', 'new_exposure_7d_woe', 'num_failed_payments_1y_woe', 'num_failed_payments_6m_woe', 'num_failed_payments_3m_woe', 'exposure_growth_woe', 'failed_payment_ratio_3m_woe', 'failed_payment_ratio_6m_woe']


In [12]:
# multicollinearity check using VIF

from statsmodels.stats.outliers_influence import variance_inflation_factor

In [15]:
vif_df = pd.DataFrame()
vif_df["feature"] = X_train.columns

vif_df["VIF"] = [
    variance_inflation_factor(X_train.values, i)
    for i in range(X_train.shape[1])
]

print(vif_df.sort_values("VIF", ascending=False))

                            feature         VIF
3                        loan_month  653.427117
4                         loan_week  612.094648
16             amount_repaid_6m_woe   19.462994
12    num_confirmed_payments_6m_woe   13.106144
20           repayment_ratio_6m_woe   12.075249
15             amount_repaid_3m_woe   10.557867
0                   is_new_customer   10.477602
14             amount_repaid_1m_woe    8.256178
11    num_confirmed_payments_3m_woe    7.811394
19              repayment_speed_woe    7.395734
17             amount_repaid_1y_woe    7.007335
2                    loan_dayofweek    5.556305
1      existing_klarna_debt_missing    5.281230
9         days_since_first_loan_woe    5.127925
13            amount_repaid_14d_woe    3.889622
18           debt_to_loan_ratio_woe    3.326454
8          existing_klarna_debt_woe    3.258606
6               payment_reliability    2.566509
10             num_active_loans_woe    2.196872
21               loan_intensity_woe    1

In [16]:
#Dropping features with high VIF

high_vif_features = vif_df.loc[vif_df["VIF"] > 8, "feature"].tolist()
X_train.drop(columns=high_vif_features, inplace=True, errors="ignore")
X_test.drop(columns=high_vif_features, inplace=True, errors="ignore")
print("dropped high vif features:", high_vif_features)

dropped high vif features: ['is_new_customer', 'loan_month', 'loan_week', 'num_confirmed_payments_6m_woe', 'amount_repaid_1m_woe', 'amount_repaid_3m_woe', 'amount_repaid_6m_woe', 'repayment_ratio_6m_woe']


In [17]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

param_grid = {
    "C": [0.01, 0.1, 1, 10],
    "class_weight": [None, "balanced", {0: 1, 1: 2}, {0: 1, 1: 3}]
}

grid = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42,solver="lbfgs"),
    param_grid=param_grid,
    scoring="recall",
    cv=5,
    
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print("Best CV recall:", grid.best_score_)

Best params: {'C': 0.01, 'class_weight': 'balanced'}
Best CV recall: 0.6423114593535748


In [18]:
log_model = LogisticRegression(
    max_iter=2000,
    class_weight='balanced',
    solver="lbfgs",
    C=0.01,
)

log_model.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",0.01
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :te

In [19]:
y_train_pred_proba = log_model.predict_proba(X_train)[:,1]
y_test_pred_proba = log_model.predict_proba(X_test)[:,1]

In [20]:
threshold = 0.4
y_test_pred = (y_test_pred_proba >= threshold).astype(int)

In [21]:
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, roc_curve

auc_train = roc_auc_score(y_train, y_train_pred_proba)
auc_test = roc_auc_score(y_test, y_test_pred_proba)

print("Train AUC:", auc_train)
print("Test AUC:", auc_test)

Train AUC: 0.6455724259570338
Test AUC: 0.6427382541416032


In [22]:
print(confusion_matrix(y_test, y_test_pred))

[[ 6370 14735]
 [  161   823]]


In [23]:
from sklearn.metrics import recall_score

recall = recall_score(y_test, y_test_pred)

print("Recall:", recall)

Recall: 0.8363821138211383


In [24]:
print(classification_report(y_test, y_test_pred))

              precision    recall  f1-score   support

           0       0.98      0.30      0.46     21105
           1       0.05      0.84      0.10       984

    accuracy                           0.33     22089
   macro avg       0.51      0.57      0.28     22089
weighted avg       0.93      0.33      0.44     22089



In [25]:
from scipy.stats import ks_2samp

ks_stat = ks_2samp(
    y_test_pred_proba[y_test == 0],
    y_test_pred_proba[y_test == 1]
)

print("KS Statistic:", ks_stat.statistic)

KS Statistic: 0.22559184333847604


In [26]:
import joblib

features_columns = X_train.columns.to_list()
threshold = 0.48

bundle = {"meta":"logistic regression model for Klarna case study",
    "model": log_model,
    "basic transforms": transform_basic,
    "merchant_encoder": merchant_encoder,
    "woe_processor": woe_processor,
    "threshold": threshold,
    "features": features_columns
}

joblib.dump(bundle, "credit_logistic_bundle.pkl")

['credit_logistic_bundle.pkl']